In [0]:
from pyspark.sql.functions import *

In [0]:
inventory=spark.read.format("json")\
    .option("multiline","true")\
    .option("inferSchema","true")\
    .option("mode","PERMISSIVE")\
    .load("s3://retail-lakehouse-ashu/raw/inventory/inventory_stock.json")

In [0]:
if "_corrupt_record" in inventory.columns:
    inventory.cache()
    inventory.count()

    corrupt_count=inventory.filter(col("_corrupt_record").isNotNull()).count()
    print(f"Corrupt Records: {corrupt_count}")
else:
    print("No _corrupt_record column present. JSON parsed successfully.")

In [0]:
inventory.printSchema()

In [0]:
inventory.display()

In [0]:
validation_condition=(
    col("products").isNotNull() &
   (size(col("products"))>0) &
    col("warehouse_id").isNotNull() &
    col("warehouse_name").isNotNull()
)

In [0]:
good_df=inventory.filter(validation_condition)
good_df.display()

In [0]:
bad_df=inventory.filter(~validation_condition)
bad_df.display()

In [0]:
bronze_inventory=good_df.withColumn("ingestion_timestamp", current_timestamp())\
    .withColumn("source_system", lit("inventory"))\
    .withColumn("source_file", col("_metadata.file_path"))\
    .withColumn("batch_id", lit(1))

bronze_inventory.display()

In [0]:
total_records=inventory.count()

good_records=good_df.count()

bad_records=bad_df.count()

print(f"Total Records   : {total_records}")
print(f"Good Records    : {good_records}")
print(f"Bad Records     : {bad_records}")

In [0]:
duplicate_df=inventory.groupBy(col("warehouse_id")).count().where(col("count")>1)
display(duplicate_df)

In [0]:
assert total_records==good_records+bad_records

In [0]:
bronze_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3://retail-lakehouse-ashu/bronze/inventory/")

In [0]:
bad_df=(
    bad_df
        .withColumn("batch_id", lit(1))
        .withColumn("pipeline_name", lit("bronze_inventory"))
        .withColumn("source_system", lit("inventory"))
        .withColumn("table_name", lit("inventory"))
        .withColumn("failure_reason", lit("MANDATORY_FIELD_VALIDATION_FAILED"))
        .withColumn("source_file", col("_metadata.file_path"))   # replace later with metadata
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("raw_record", to_json(struct(*bad_df.columns)))
        .select(
        col("batch_id"),
        col("pipeline_name"),
        col("source_system"),
        col("table_name"),
        col("failure_reason"),
        col("source_file"),
        col("ingestion_timestamp"),
        col("raw_record")
    )
)

In [0]:
bad_df.write \
    .format("delta") \
    .mode("append") \
    .save("s3://retail-lakehouse-ashu/bad_records/")